In [1]:
%%writefile main.py
import math
import time
import numpy as np
from collections import defaultdict

# ============================================================
# Engine Configuration & Strategy Profile
# ============================================================
BOARD = 100.0
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY_MARGIN = 0.5 
ROTATION_LIMIT = 50.0
HORIZON = 100
TOTAL_STEPS = 500

#[WINNER] Logistics Profile Settings
SHORT_RANGE_THRESHOLD = 20    # Max turns to qualify as a "Short-Range Strike"
LONG_RANGE_PENALTY = 0.35     # 65% reduction in ROI for long-range snipes
CARRIER_ALIGNMENT_BONUS = 1.8 # Multiplier for striking while rotating towards target
CARRIER_DELTA_WINDOW = 5      # Turns to project trajectory alignment

# ============================================================
# Vectorized Physics & Tensors
# ============================================================

def speed_func_scalar(ships):
    if ships <= 1: return 1.0
    ratio = max(0.0, min(1.0, math.log(ships) / math.log(1000.0)))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)

def check_sun_collision(sx, sy, tx, ty):
    """Fast linear algebra segment-to-point distance for Sun avoidance."""
    dx, dy = tx - sx, ty - sy
    l2 = dx*dx + dy*dy
    if l2 == 0: return False
    t = max(0.0, min(1.0, ((CENTER_X - sx)*dx + (CENTER_Y - sy)*dy) / l2))
    proj_x = sx + t * dx
    proj_y = sy + t * dy
    return math.hypot(CENTER_X - proj_x, CENTER_Y - proj_y) <= SUN_R + SUN_SAFETY_MARGIN

def build_world_tensor(planets, ang_vel, comets, comet_ids):
    """
    CPU Optimization: Precomputes the exact (X, Y) of every planet for the next 100 turns.
    Returns: pos_arr of shape (Max_Planet_ID + 1, HORIZON, 2)
    """
    max_id = max(p[0] for p in planets) if planets else 0
    max_id = max(max_id, max((max(c["planet_ids"]) for c in comets if c["planet_ids"]), default=0))
    
    pos_arr = np.full((max_id + 1, HORIZON, 2), np.nan)
    radii = np.zeros(max_id + 1)
    
    for p in planets:
        pid, _, px, py, pr, _, _ = p
        radii[pid] = pr
        
        if pid in comet_ids:
            for group in comets:
                if pid in group["planet_ids"]:
                    idx = group["planet_ids"].index(pid)
                    paths = group["paths"]
                    start_idx = group.get("path_index", 0)
                    for t in range(HORIZON):
                        f_idx = start_idx + t
                        if idx < len(paths) and 0 <= f_idx < len(paths[idx]):
                            pos_arr[pid, t, 0] = paths[idx][f_idx][0]
                            pos_arr[pid, t, 1] = paths[idx][f_idx][1]
        else:
            orb_r = math.hypot(px - CENTER_X, py - CENTER_Y)
            if orb_r + pr >= ROTATION_LIMIT:
                # Static Planet
                pos_arr[pid, :, 0] = px
                pos_arr[pid, :, 1] = py
            else:
                # Orbiting Planet (Vectorized projection)
                curr_ang = math.atan2(py - CENTER_Y, px - CENTER_X)
                angles = curr_ang + ang_vel * np.arange(HORIZON)
                pos_arr[pid, :, 0] = CENTER_X + orb_r * np.cos(angles)
                pos_arr[pid, :, 1] = CENTER_Y + orb_r * np.sin(angles)
                
    return pos_arr, radii

def get_intercept_tensor(src_id, tgt_id, ships, pos_arr, radii):
    """Fast Array-Scan to find exactly which frame the fleet impacts the target."""
    speed = speed_func_scalar(ships)
    S_pos = pos_arr[src_id, 0] # Source current location
    T_pos = pos_arr[tgt_id]    # Target location array (HORIZON, 2)
    
    # Distance from Source to Target at every future frame
    dist_to_T = np.hypot(T_pos[:, 0] - S_pos[0], T_pos[:, 1] - S_pos[1])
    
    # Distance required to travel (Surface to Surface)
    req_dist = np.maximum(0, dist_to_T - radii[src_id] - radii[tgt_id])
    avail_dist = speed * np.arange(HORIZON)
    
    # Find the exact frame where Available Distance overtakes Required Distance
    valid_frames = np.where(avail_dist >= req_dist)[0]
    
    for t in valid_frames:
        if t == 0 or np.isnan(T_pos[t, 0]): continue
        tx, ty = T_pos[t]
        if not check_sun_collision(S_pos[0], S_pos[1], tx, ty):
            angle = math.atan2(ty - S_pos[1], tx - S_pos[0])
            return angle, t
    return None, None

def build_fleet_ledger(fleets, pos_arr, radii):
    """Vectorized flight resolution for currently active fleets."""
    ledger = defaultdict(list)
    for f in fleets:
        fid, owner, fx, fy, angle, _, ships = f
        speed = speed_func_scalar(ships)
        dx, dy = math.cos(angle), math.sin(angle)
        
        t_arr = np.arange(HORIZON)
        Fx = fx + speed * t_arr * dx
        Fy = fy + speed * t_arr * dy
        
        best_t, best_pid = 999, None
        
        for pid in range(pos_arr.shape[0]):
            if np.isnan(pos_arr[pid, 0, 0]): continue
            # Distance from moving fleet to moving planet at every frame
            dist_arr = np.hypot(pos_arr[pid, :, 0] - Fx, pos_arr[pid, :, 1] - Fy)
            hits = np.where(dist_arr <= radii[pid])[0]
            if len(hits) > 0 and hits[0] < best_t:
                best_t = hits[0]
                best_pid = pid
                
        if best_pid is not None:
            ledger[best_pid].append((best_t, owner, int(ships)))
            
    return ledger

# ============================================================
# Logistics State Evaluation
# ============================================================

def evaluate_planet_timeline(pid, owner, garrison, prod, ledger_entries, horizon, player_id):
    """Calculates who will own the planet, when, and how many ships it will have."""
    by_turn = defaultdict(list)
    for t, o, s in ledger_entries:
        if t <= horizon: by_turn[t].append((o, s))
        
    curr_owner = owner
    curr_ships = float(garrison)
    
    for turn in range(1, horizon + 1):
        if curr_owner != -1: curr_ships += prod
        if turn in by_turn:
            arrivals = defaultdict(int)
            for a_o, a_s in by_turn[turn]: arrivals[a_o] += a_s
            
            if arrivals:
                # Sort arriving fleets by size
                sorted_arr = sorted(arrivals.items(), key=lambda x: x[1], reverse=True)
                top_o, top_s = sorted_arr[0]
                surv_o, surv_s = top_o, top_s
                
                if len(sorted_arr) > 1:
                    sec_s = sorted_arr[1][1]
                    if top_s == sec_s: surv_o, surv_s = -1, 0
                    else: surv_s -= sec_s
                
                if surv_s > 0:
                    if curr_owner == surv_o:
                        curr_ships += surv_s
                    else:
                        curr_ships -= surv_s
                        if curr_ships < 0:
                            curr_owner = surv_o
                            curr_ships = -curr_ships
                            
    return curr_owner, curr_ships

# ============================================================
# Main CPU Agent Engine
# ============================================================

def agent(obs, config=None):
    is_dict = isinstance(obs, dict)
    player = obs.get("player", 0) if is_dict else getattr(obs, "player", 0)
    step = obs.get("step", 0) if is_dict else getattr(obs, "step", 0)
    
    raw_planets = obs.get("planets",[]) if is_dict else getattr(obs, "planets",[])
    raw_fleets = obs.get("fleets", []) if is_dict else getattr(obs, "fleets",[])
    ang_vel = obs.get("angular_velocity", 0.0) if is_dict else getattr(obs, "angular_velocity", 0.0)
    comets = obs.get("comets",[]) if is_dict else getattr(obs, "comets",[])
    comet_ids = set(obs.get("comet_planet_ids",[]) if is_dict else getattr(obs, "comet_planet_ids",[]))
    
    rem_steps = max(1, TOTAL_STEPS - step)
    
    # 1. Linear-Scan Precomputation
    pos_arr, radii = build_world_tensor(raw_planets, ang_vel, comets, comet_ids)
    ledger = build_fleet_ledger(raw_fleets, pos_arr, radii)
    
    my_planets = []
    other_planets =[]
    
    for p in raw_planets:
        pid, owner, px, py, pr, ships, prod = p
        # Quick timeline resolve
        proj_owner, proj_ships = evaluate_planet_timeline(pid, owner, ships, prod, ledger.get(pid,[]), 20, player)
        
        # Dictionary format for easy access
        p_data = {"id": pid, "owner": owner, "proj_owner": proj_owner, "proj_ships": proj_ships, 
                  "ships": ships, "prod": prod, "is_comet": pid in comet_ids}
        
        if owner == player:
            my_planets.append(p_data)
        else:
            other_planets.append(p_data)

    if not my_planets: return[]

    # 2. Greedy ROI Matrix Generation
    roi_matrix = []
    budget = {p["id"]: p["ships"] for p in my_planets}
    
    for mine in my_planets:
        src_id = mine["id"]
        avail_ships = budget[src_id]
        if avail_ships <= 5: continue
        
        for tgt in other_planets:
            tgt_id = tgt["id"]
            
            # Fast heuristic distance check (Turn 0 distance)
            dist_t0 = np.hypot(pos_arr[tgt_id, 0, 0] - pos_arr[src_id, 0, 0], 
                               pos_arr[tgt_id, 0, 1] - pos_arr[src_id, 0, 1])
            approx_T = int(dist_t0 / speed_func_scalar(avail_ships))
            
            if approx_T > min(rem_steps, 50): continue # Ignore impossible distances
            
            # Predict Enemy Cost Accounting for Transit Accrual
            cost = tgt["proj_ships"]
            if tgt["proj_owner"] not in (-1, player):
                cost += tgt["prod"] * approx_T
                
            cost += 2 # Margin of safety
            
            if avail_ships < cost: continue # We can't afford it
            
            # Metrics Extraction: The "Carrier Delta"
            # Determines if this specific planet is acting as a "Forward Aircraft Carrier"
            dist_t5 = np.hypot(pos_arr[tgt_id, min(CARRIER_DELTA_WINDOW, HORIZON-1), 0] - pos_arr[src_id, min(CARRIER_DELTA_WINDOW, HORIZON-1), 0], 
                               pos_arr[tgt_id, min(CARRIER_DELTA_WINDOW, HORIZON-1), 1] - pos_arr[src_id, min(CARRIER_DELTA_WINDOW, HORIZON-1), 1])
            
            carrier_delta = dist_t0 - dist_t5 # Positive means we are moving closer together
            
            # THE GREEDY ROI FORMULA
            # ROI = (Production * Remaining_Time) / Cost
            roi = ((tgt["prod"] + 0.1) * max(0, rem_steps - approx_T)) / (cost + 1.0)
            
            # Apply Logistics Multipliers
            if carrier_delta > 0:
                roi *= CARRIER_ALIGNMENT_BONUS # Strike while the iron is hot!
            
            if approx_T <= SHORT_RANGE_THRESHOLD:
                roi *= 1.2 # [WINNER] Profile: Heavily favor short range strikes
            else:
                roi *= LONG_RANGE_PENALTY # Penalize deep-space debt
                
            if tgt["is_comet"]:
                roi *= 0.6 # Comets expire, inherently lower ROI
                
            roi_matrix.append((roi, src_id, tgt_id, int(cost)))
            
    # 3. CPU Sort & Execute
    # Sort matrix perfectly highest ROI to lowest
    roi_matrix.sort(key=lambda x: x[0], reverse=True)
    
    moves =[]
    
    for roi, src_id, tgt_id, cost in roi_matrix:
        # Check if the "Carrier" still has ships in its budget
        if budget[src_id] >= cost:
            
            # 4. Precision Frame-Perfect Intercept Verification
            # Only do the heavy math for the top ROI candidates that we can actually afford!
            angle, arrival_t = get_intercept_tensor(src_id, tgt_id, cost, pos_arr, radii)
            
            if angle is not None:
                # Add to dispatch
                moves.append([src_id, angle, cost])
                
                # Deduct from bank
                budget[src_id] -= cost
                
                # Update Ledger state internally to prevent double-spending
                ledger[tgt_id].append((arrival_t, player, cost))

    return moves

Writing main.py
